# Cancel In-Progress Fabric Jobs

Self-contained notebook for Microsoft Fabric. Runs inside a Fabric
notebook with the notebook's own identity - no `az login` or token
handling needed.

**What it does**

1. Lists workspaces (filtered by capacity or name, or all accessible).
2. For each workspace, lists items and queries their job instances.
3. Filters to `status == "InProgress"`.
4. Cancels each one and verifies it reaches a terminal state.

**Scales to large tenants** (1000+ workspaces) via:
- `capacity_filter` to scope to specific Fabric capacities
- parallel per-workspace scanning
- a safety gate that refuses unfiltered scans of giant tenants

**Self-protection:** the notebook never cancels itself.

**Throttling:** honours `Retry-After` on HTTP 429 responses.

Run cells top-to-bottom.


## 1. Discover every in-progress job

**Scoping options** (edit the variables at the top of cell 3):

```python
# Option A: scan ONE specific workspace
workspace_filter = ["MyWorkspaceName"]

# Option B: scan SEVERAL specific workspaces
workspace_filter = ["Workspace1", "Workspace2", "Workspace3"]

# Option C: scan every workspace on a specific Fabric capacity
capacity_filter = ["MyCapacityName"]            # or by GUID

# Option D: scan everything the user can see
#          (auto-aborts above MAX_AUTO_WORKSPACES = 50)
workspace_filter = []
capacity_filter  = []
```

Workspace and capacity names are **case-insensitive**. You can also pass
GUIDs in `capacity_filter` (it accepts both display name and ID).

Outputs a `pandas.DataFrame` called `df` with one row per `InProgress`
job. Cell 2 (cancel) reads `df`.


In [ ]:
import sempy.fabric as fabric
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

client = fabric.FabricRestClient()
FABRIC_API = "https://api.fabric.microsoft.com/v1"

# ---------------- Scoping (edit before running) -----------------------------
# To target specific workspaces, list their display names:
#     workspace_filter = ["MyWorkspace"]
#     workspace_filter = ["WS-A", "WS-B", "WS-C"]
# To target all workspaces on a Fabric capacity (name OR GUID):
#     capacity_filter = ["MyCapacityName"]
# Leave both empty to scan every workspace the user can see (aborts above
# MAX_AUTO_WORKSPACES to avoid 1000+-workspace fanout in large tenants).
workspace_filter: list[str] = []   # display names of specific workspaces
capacity_filter:  list[str] = []   # capacity display names OR GUIDs
WORKSPACE_PARALLELISM = 8           # workspaces to scan concurrently
MAX_AUTO_WORKSPACES   = 50          # safety: abort if scanning more than N
NO_JOBS_TYPES = {"SQLEndpoint", "Dashboard", "PaginatedReport"}
# ----------------------------------------------------------------------------


def get_all_pages(url: str) -> list[dict]:
    """GET with continuation-uri pagination and Retry-After-aware 429 retry."""
    rows = []
    while url:
        response = client.get(url)
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", "30"))
            print(f"Throttled. Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue
        if response.status_code not in (200, 202):
            raise Exception(f"GET {url} failed: {response.status_code} - {response.text}")
        data = response.json()
        rows.extend(data.get("value", []))
        url = data.get("continuationUri")
    return rows


def get_self_context() -> dict:
    """Identify the notebook running this code so we never cancel ourselves."""
    try:
        from notebookutils import mssparkutils  # type: ignore
        ctx = getattr(mssparkutils.runtime, "context", {}) or {}
        return {
            "notebook_id":
                ctx.get("currentNotebookId") or ctx.get("notebookId") or "",
            "workspace_id":
                ctx.get("currentWorkspaceId") or ctx.get("workspaceId") or "",
            "job_instance_id":
                ctx.get("activityId") or ctx.get("jobInstanceId")
                or ctx.get("runId") or "",
        }
    except Exception:
        return {}


SELF = get_self_context()
if SELF.get("notebook_id"):
    print(f"Self-protect: excluding own notebook {SELF['notebook_id'][:8]}...")


# 1. Enumerate workspaces the user can see
print("Listing accessible workspaces...")
all_workspaces = get_all_pages(f"{FABRIC_API}/workspaces")
all_workspaces = [ws for ws in all_workspaces if ws.get("type") == "Workspace"]
print(f"  user has access to {len(all_workspaces)} workspace(s).")


# 2. Apply capacity / workspace filters
_CAP_NAMES: dict[str, str] = {}


def _match_capacity(ws: dict, keys: set) -> bool:
    """Accept either capacity GUID or display name (case-insensitive)."""
    cap_id = (ws.get("capacityId") or "").lower()
    if not cap_id:
        return False
    if cap_id in keys:
        return True
    if cap_id not in _CAP_NAMES:
        try:
            r = client.get(f"{FABRIC_API}/capacities/{cap_id}")
            _CAP_NAMES[cap_id] = (r.json().get("displayName") or "").lower() \
                if r.status_code == 200 else ""
        except Exception:
            _CAP_NAMES[cap_id] = ""
    return _CAP_NAMES[cap_id] in keys


if workspace_filter:
    wanted = {n.lower() for n in workspace_filter}
    workspaces = [ws for ws in all_workspaces
                  if (ws.get("displayName") or "").lower() in wanted]
    print(f"  workspace_filter -> {len(workspaces)} match(es).")
elif capacity_filter:
    keys = {c.lower() for c in capacity_filter}
    workspaces = [ws for ws in all_workspaces if _match_capacity(ws, keys)]
    print(f"  capacity_filter  -> {len(workspaces)} workspace(s) on the named capacities.")
else:
    workspaces = all_workspaces
    if len(workspaces) > MAX_AUTO_WORKSPACES:
        print()
        print(f"!!! {len(workspaces)} workspaces is a lot.")
        print(f"!!! Scanning all of them will take a long time and may hit 429 throttling.")
        print(f"!!! Recommended: set capacity_filter or workspace_filter at the top of")
        print(f"!!! this cell and re-run. Examples:")
        print(f"!!!   capacity_filter  = ['my-overloaded-capacity']")
        print(f"!!!   workspace_filter = ['ws-1', 'ws-2']")
        print(f"!!! To proceed anyway, raise MAX_AUTO_WORKSPACES above {len(workspaces)}.")
        raise SystemExit("Aborted: too many workspaces to auto-scan.")

print(f"Scanning {len(workspaces)} workspace(s) with parallelism={WORKSPACE_PARALLELISM}...")
t0 = time.time()


def scan_one_workspace(ws: dict) -> list[dict]:
    workspace_id = ws["id"]
    workspace_name = ws["displayName"]
    rows: list[dict] = []
    try:
        items = get_all_pages(f"{FABRIC_API}/workspaces/{workspace_id}/items")
    except Exception as e:
        print(f"  ! {workspace_name}: list items failed - {e}")
        return rows
    for item in items:
        if item.get("type") in NO_JOBS_TYPES:
            continue
        item_id = item.get("id")
        try:
            jobs = get_all_pages(
                f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}/jobs/instances"
            )
        except Exception:
            continue
        for job in jobs:
            if job.get("status") != "InProgress":
                continue
            if item_id == SELF.get("notebook_id") or \
                    job.get("id") == SELF.get("job_instance_id"):
                continue
            rows.append({
                "workspaceName":  workspace_name,
                "workspaceId":    workspace_id,
                "itemName":       item.get("displayName"),
                "itemType":       item.get("type"),
                "itemId":         item_id,
                "jobInstanceId":  job.get("id"),
                "jobType":        job.get("jobType"),
                "invokeType":     job.get("invokeType"),
                "status":         job.get("status"),
                "startTimeUtc":   job.get("startTimeUtc"),
                "rootActivityId": job.get("rootActivityId"),
            })
    return rows


running_jobs: list[dict] = []
done = 0
step = max(1, len(workspaces) // 20)
with ThreadPoolExecutor(max_workers=WORKSPACE_PARALLELISM) as pool:
    futs = {pool.submit(scan_one_workspace, ws): ws for ws in workspaces}
    for f in as_completed(futs):
        running_jobs.extend(f.result())
        done += 1
        if done % step == 0 or done == len(workspaces):
            print(f"  {done}/{len(workspaces)} workspaces scanned, "
                  f"{len(running_jobs)} active jobs found "
                  f"(elapsed {time.time()-t0:.0f}s)")

print(f"\nDone in {time.time()-t0:.0f}s. Found {len(running_jobs)} in-progress job(s).")

df = pd.DataFrame(running_jobs)
if df.empty:
    print("No in-progress jobs found.")
else:
    display(df.sort_values(["workspaceName", "startTimeUtc"]))


## 2. Cancel every in-progress job

Issues `POST /jobs/instances/{id}/cancel` for every row of `df`. After
cancelling, polls each instance for up to 60 s until it reaches a
terminal status (`Cancelled` / `Completed` / `Failed`).


In [ ]:
def cancel_job(workspace_id: str, item_id: str, instance_id: str) -> tuple[int, str]:
    """POST /jobs/instances/{id}/cancel with Retry-After-aware 429 retry."""
    url = (f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}"
           f"/jobs/instances/{instance_id}/cancel")
    for attempt in range(6):
        r = client.post(url)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "5"))
            print(f"  Throttled, waiting {wait}s (attempt {attempt+1}/6)")
            time.sleep(wait)
            continue
        return r.status_code, (r.text or "").strip()
    return r.status_code, (r.text or "").strip()


def get_job_status(workspace_id: str, item_id: str, instance_id: str) -> str:
    r = client.get(f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}"
                   f"/jobs/instances/{instance_id}")
    if r.status_code != 200:
        return f"error:{r.status_code}"
    return r.json().get("status", "Unknown")


if df.empty:
    print("Nothing to cancel.")
else:
    print(f"Cancelling {len(df)} in-progress job(s) (parallelism=8)...")
    cancelled: list[dict] = []

    def _do_cancel(row):
        code, body = cancel_job(row.workspaceId, row.itemId, row.jobInstanceId)
        return row, code, body

    with ThreadPoolExecutor(max_workers=8) as pool:
        futs = [pool.submit(_do_cancel, row) for row in df.itertuples()]
        for f in as_completed(futs):
            row, code, body = f.result()
            ok = 200 <= code < 300
            marker = "OK  " if ok else "FAIL"
            print(f"  {marker} [{code}]  {row.workspaceName} / {row.itemName}  ({row.jobInstanceId})")
            if ok:
                cancelled.append({
                    "workspaceId":   row.workspaceId,
                    "itemId":        row.itemId,
                    "jobInstanceId": row.jobInstanceId,
                    "label":         f"{row.workspaceName} / {row.itemName}",
                })

    # Verify terminal status
    TERMINAL = {"Completed", "Failed", "Cancelled", "Deduped"}
    print(f"\nPolling {len(cancelled)} cancel(s) for terminal status (up to 60s)...")
    deadline = time.time() + 60
    pending = {c["jobInstanceId"]: c for c in cancelled}
    while pending and time.time() < deadline:
        time.sleep(5)
        for inst_id in list(pending):
            c = pending[inst_id]
            status = get_job_status(c["workspaceId"], c["itemId"], inst_id)
            if status in TERMINAL:
                print(f"  {c['label']}  ->  {status}")
                pending.pop(inst_id, None)
    for inst_id, c in pending.items():
        status = get_job_status(c["workspaceId"], c["itemId"], inst_id)
        print(f"  {c['label']}  ->  {status} (still pending after 60s)")


## 3. Continuous monitoring (optional)

Keeps re-scanning + cancelling new in-progress jobs every
`POLL_INTERVAL_SECONDS` for `LOOP_DURATION_MINUTES`. Use to drain a
runaway capacity. Stop the cell to exit early.

Re-uses the `workspace_filter` / `capacity_filter` you set in cell 1.


In [ ]:
LOOP_DURATION_MINUTES = 30
POLL_INTERVAL_SECONDS = 30

loop_started_at = time.time()
loop_ends_at = loop_started_at + LOOP_DURATION_MINUTES * 60
iteration = 0
totals = {"found": 0, "cancelled": 0}

print(f"Continuous monitoring for {LOOP_DURATION_MINUTES} min, "
      f"rescan every {POLL_INTERVAL_SECONDS}s")
print(f"(Using the same workspaces selected in cell 1: {len(workspaces)} workspace(s).)")

while time.time() < loop_ends_at:
    iteration += 1
    remaining = int(loop_ends_at - time.time())
    print(f"\n=== Iteration {iteration}  (remaining: {remaining}s) ===")

    running: list[dict] = []
    with ThreadPoolExecutor(max_workers=WORKSPACE_PARALLELISM) as pool:
        futs = [pool.submit(scan_one_workspace, ws) for ws in workspaces]
        for f in as_completed(futs):
            running.extend(f.result())

    print(f"  found {len(running)} active job(s)")
    totals["found"] += len(running)

    if running:
        with ThreadPoolExecutor(max_workers=8) as pool:
            futs = [pool.submit(cancel_job, r["workspaceId"], r["itemId"], r["jobInstanceId"])
                    for r in running]
            for r, fut in zip(running, futs):
                code, body = fut.result()
                ok = 200 <= code < 300
                if ok:
                    totals["cancelled"] += 1
                print(f"    {'OK  ' if ok else 'FAIL'} [{code}]  "
                      f"{r['workspaceName']} / {r['itemName']}  ({r['jobInstanceId']})")

    if time.time() >= loop_ends_at:
        break
    sleep_s = min(POLL_INTERVAL_SECONDS, max(0, int(loop_ends_at - time.time())))
    if sleep_s:
        print(f"  sleeping {sleep_s}s before next iteration...")
        time.sleep(sleep_s)

print(f"\nLoop complete. Totals: {totals}")
